# Generate Synthetic Data: Depth-Dependent XPS

This notebook generates the synthetic 2D dataset used by `example.ipynb`.
It saves the data to `data/` as CSV files.

**Physics:**
A single GLP (Gaussian-Lorentzian Product) core-level peak is measured with
time-resolved XPS after laser excitation. Two depth profiles are combined:

1. **IMFP amplitude profile**: XPS is surface-sensitive; signal from depth *z*
   decays as `A_0 * exp(-z / lambda)` where lambda is the inelastic mean free path.
2. **Band bending position profile**: Surface electric field shifts the core level
   binding energy linearly with depth: `x0(z) = x0_surface + m * z`.

**Time dependence (spectral diffusion):**
After laser excitation a surface photovoltage partially collapses the band bending.
The gradient `m` recovers exponentially with time constant tau_bb.

In [ ]:
import os

import numpy as np
from scipy.special import erf

## Define axes and true physical parameters

In [ ]:
# --- Axes ---
energy = np.linspace(95.0, 102.0, 210)  # binding energy (eV)
time_ax = np.linspace(-50, 300, 176)  # time delay (ps)
depth = np.linspace(0, 10, 50)  # probing depth (nm)

# --- True physical parameters (what we aim to recover in example.ipynb) ---
a_surface = 100.0  # surface spectral weight
tau_imfp = 2.0  # IMFP (nm)
x0_surface = 99.5  # surface binding energy (eV)
m_bb_eq = -0.5  # equilibrium band bending gradient (eV/nm)
a_bb_collapse = 0.200  # amplitude of transient BB collapse (eV/nm)
tau_bb = 100.0  # BB recovery time constant (ps)
sigma_irf = 10.0  # Gaussian IRF width (ps)
f_peak = 1.2  # GLP FWHM (eV)
m_glp = 0.3  # Lorentzian fraction
y0_bg = 0.5  # flat background
noise_level = 0.3  # Gaussian noise std

## Build 2D dataset

In [ ]:
def glp(e, A, x0, F, m):
    """Gaussian-Lorentzian Product peak."""
    sigma = F / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    gauss = np.exp(-0.5 * ((e - x0) / sigma) ** 2)
    lorentz = 1.0 / (1.0 + ((e - x0) / (0.5 * F)) ** 2)
    return A * ((1 - m) * gauss + m * lorentz)


def depth_avg_spectrum(e, depth, A_s, tau_l, x0_s, m_bb, F, m_g, y0):
    """Compute depth-integrated XPS spectrum with IMFP weighting and band bending."""
    spectrum = np.zeros_like(e)
    for z in depth:
        a_z = A_s * np.exp(-z / tau_l)  # IMFP weighting
        x0_z = x0_s + m_bb * z  # band bending shift
        spectrum += glp(e, a_z, x0_z, F, m_g)
    return spectrum / len(depth) + y0


# --- Build 2D dataset ---
rng = np.random.default_rng(42)
data = np.zeros((len(time_ax), len(energy)))

for t_i, t in enumerate(time_ax):
    # band bending gradient at this time step:
    # smooth onset (Gaussian IRF) + exponential recovery
    heaviside = 0.5 * (1.0 + erf(t / (sigma_irf * np.sqrt(2.0))))
    m_bb_t = m_bb_eq + a_bb_collapse * heaviside * np.exp(
        -max(t, 0) / tau_bb
    )
    data[t_i] = depth_avg_spectrum(
        energy, depth, a_surface, tau_imfp,
        x0_surface, m_bb_t, f_peak, m_glp, y0_bg,
    )

data += rng.normal(0, noise_level, data.shape)

print(f"Data shape : {data.shape}  (time x energy)")
print(f"Energy     : {energy.min():.1f} - {energy.max():.1f} eV")
print(f"Time       : {time_ax.min():.0f} - {time_ax.max():.0f} ps")
print(f"Depth axis : {depth.min():.0f} - {depth.max():.0f} nm  ({len(depth)} points)")

## Save to CSV

In [ ]:
data_dir = os.path.join(os.getcwd())
os.makedirs(data_dir, exist_ok=True)

np.savetxt(os.path.join(data_dir, "data.csv"), data, delimiter=",")
np.savetxt(os.path.join(data_dir, "energy.csv"), energy, delimiter=",")
np.savetxt(os.path.join(data_dir, "time.csv"), time_ax, delimiter=",")
np.savetxt(os.path.join(data_dir, "aux_axis.csv"), depth, delimiter=",")

print(f"Saved to {data_dir}/")
for f in ["data.csv", "energy.csv", "time.csv", "aux_axis.csv"]:
    size = os.path.getsize(os.path.join(data_dir, f))
    print(f"  {f:15s} {size / 1024:.1f} kB")